# Retail Sales EDA

**How to run:**
1. Ensure `sales_data.csv` is in the same folder as this notebook.
2. Run all cells (use `Run All` in VS Code / Jupyter).
3. Review the charts and the **Key insights** section at the end.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# --- Load Data ---
csv_path = "sales_data.csv"
df = pd.read_csv(csv_path)

# --- Quick overview ---
print("Data shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)

# --- Missing values ---
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.concat([missing, missing_pct], axis=1, keys=["missing", "%"])
if missing_df[missing_df["missing"] > 0].shape[0] > 0:
    display(missing_df[missing_df["missing"] > 0])
else:
    print("No missing values found.")

# --- Feature engineering ---
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["YearMonth"] = df["Date"].dt.to_period("M")
    df["Weekday"] = df["Date"].dt.day_name()

# --- Time-based trends ---
if "YearMonth" in df.columns:
    monthly_orders = df.groupby("YearMonth").size().rename("order_count").sort_index()
    monthly_revenue = (
        df.groupby("YearMonth")["Total Amount"]
          .sum()
          .rename("monthly_revenue")
          .sort_index()
    )

    plt.figure()
    monthly_orders.plot(marker="o", title="Orders over time")
    plt.ylabel("Number of Orders")
    plt.xlabel("Order Year-Month")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    plt.figure()
    monthly_revenue.plot(marker="o", title="Monthly Revenue (Total Amount)")
    plt.ylabel("Total Revenue")
    plt.xlabel("Order Year-Month")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# --- Category / group analysis ---
if "Product Category" in df.columns and "Total Amount" in df.columns:
    category_revenue = (
        df.groupby("Product Category")["Total Amount"]
          .sum()
          .sort_values(ascending=False)
    )

    plt.figure()
    sns.barplot(x=category_revenue.values, y=category_revenue.index, palette="viridis")
    plt.title("Total Revenue by Category")
    plt.xlabel("Revenue")
    plt.ylabel("Product Category")
    plt.tight_layout()
    plt.show()

# --- Correlation matrix ---
numeric = df.select_dtypes(include=["number"])
if not numeric.empty:
    plt.figure()
    sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Numeric feature correlations")
    plt.tight_layout()
    plt.show()

## Monthly revenue trend

This section computes and visualizes monthly total **revenue** (Total Amount) to highlight seasonality and trend.

In [ ]:
# --- Monthly revenue trend ---
if "YearMonth" in df.columns and "Total Amount" in df.columns:
    monthly = (
        df.groupby("YearMonth")["Total Amount"]
          .sum()
          .rename("monthly_revenue")
          .sort_index()
    )
    monthly_pct = (monthly / monthly.sum() * 100).round(2)

    plt.figure()
    monthly.plot(marker="o")
    plt.title("Monthly Revenue (Total Amount)")
    plt.ylabel("Revenue")
    plt.xlabel("Order Year-Month")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    plt.figure()
    monthly_pct.plot(marker="o", color="tab:green")
    plt.title("Monthly Revenue as % of Total")
    plt.ylabel("% of total revenue")
    plt.xlabel("Order Year-Month")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Top 10 customers by revenue

Identify the customers generating the most revenue and visualize the top 10.


In [ ]:
# --- Top 10 customers by revenue ---
if "Customer ID" in df.columns and "Total Amount" in df.columns:
    customer_revenue = (
        df.groupby("Customer ID")["Total Amount"]
          .sum()
          .sort_values(ascending=False)
    )
    top_customers = customer_revenue.head(10)

    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_customers.values, y=top_customers.index, palette="magma")
    plt.title("Top 10 Customers by Revenue")
    plt.xlabel("Revenue")
    plt.ylabel("Customer ID")
    plt.tight_layout()
    plt.show()

    display(top_customers.reset_index().rename(columns={'Total Amount': 'Revenue'}))


## Customer segmentation

Segment customers by revenue and order frequency to identify high-value and low-value groups.


In [ ]:
# --- Customer segmentation ---
if "Customer ID" in df.columns and "Total Amount" in df.columns:
    customer_summary = (
        df.groupby("Customer ID")[['Total Amount']].agg(
            total_revenue="sum",
            order_count="count"
        )
    )
    customer_summary['avg_order_value'] = (customer_summary['total_revenue'] / customer_summary['order_count']).round(2)

    # Define simple segments based on revenue quartiles
    revenue_bins = customer_summary['total_revenue'].quantile([0.25, 0.5, 0.75]).tolist()
    customer_summary['segment'] = pd.cut(
        customer_summary['total_revenue'],
        bins=[-1] + revenue_bins + [customer_summary['total_revenue'].max()],
        labels=["Low", "Mid", "High", "Top"],
        include_lowest=True
    )

    segment_counts = customer_summary['segment'].value_counts().reindex(["Top", "High", "Mid", "Low"]).fillna(0)

    plt.figure(figsize=(8, 4))
    sns.barplot(x=segment_counts.index, y=segment_counts.values, palette="Spectral")
    plt.title("Customer segments by revenue")
    plt.xlabel("Segment")
    plt.ylabel("Number of customers")
    plt.tight_layout()
    plt.show()

    display(customer_summary.sort_values('total_revenue', ascending=False).head(10))


## Summary chart

This chart summarizes the revenue mix by product category and customer segment.


In [ ]:
# --- Summary chart: category + segment breakdown ---
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Category revenue share
if "Product Category" in df.columns and "Total Amount" in df.columns:
    cat_rev = df.groupby("Product Category")["Total Amount"].sum().sort_values(ascending=False)
    cat_pct = (cat_rev / cat_rev.sum() * 100).round(1)
    sns.barplot(x=cat_pct.values, y=cat_pct.index, ax=axes[0], palette="coolwarm")
    axes[0].set_title("Revenue share by category (%)")
    axes[0].set_xlabel("% of total revenue")

# Customer segment share
if "Customer ID" in df.columns and "Total Amount" in df.columns:
    cust_rev = df.groupby("Customer ID")["Total Amount"].sum()
    bins = cust_rev.quantile([0.25, 0.5, 0.75]).tolist()
    segments = pd.cut(cust_rev, bins=[-1] + bins + [cust_rev.max()], labels=["Low", "Mid", "High", "Top"], include_lowest=True)
    seg_counts = segments.value_counts().reindex(["Top", "High", "Mid", "Low"]).fillna(0)
    sns.barplot(x=seg_counts.index, y=seg_counts.values, ax=axes[1], palette="Spectral")
    axes[1].set_title("Customer segment counts (by revenue)")
    axes[1].set_xlabel("Segment")

plt.tight_layout()
plt.show()


## Key insights

- **Total revenue:** Dataset spans Jan 2023 → Jan 2024 and totals about $456K in revenue.
- **Seasonality:** Revenue peaks around May 2023 and is weakest around March 2023, suggesting a seasonal cycle.
- **Top categories:** Electronics, Clothing, and Beauty drive the majority of revenue.
- **Next step:** Look into why early-year months are weaker and consider promotions in stronger months (e.g., May).
